# Gear 2.2 research — signal→fill spread response

**Research only.** Не канон `model_gear2` / `VARIATION` / `HYPER`. PnL не критерий.

Гипотеза: для больших положительных \(x_t=s_{signal}-F_t\) изменение за `Trade_Lat` направлено к схлопыванию: \(\mathrm{corr}(x,\Delta_{100})<0\).

Альтернатива: растёт только \(|\Delta|\), медиана направленного \(\Delta\) ~ 0.

Не утверждаем причину (арб-боты): L1 показывает mean reversion / stale-leg / async discovery, не участника.


In [ ]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from research.gear22_signal_fill_lib import (
    AUGUST_SEGMENTS,
    OUT_DIR,
    PRIMARY_L,
    summarize,
    verdict_from_summary,
    spearman_corr,
)

REPO = Path(".").resolve()
OUT = OUT_DIR
EVENTS = OUT / "events.parquet"
SMOKE = OUT / "smoke" / "events.parquet"

print("segments (UTC, END exclusive) from docs/model-data-coverage.md:")
for a, b in AUGUST_SEGMENTS:
    print(f"  {a} → {b}")
print("OUT", OUT)
print("events exists", EVENTS.exists(), "smoke", SMOKE.exists())


## Load events + pre-registered freezes

`F_t` = past-only median (5 min); \(\sigma=1.4826\cdot MAD\); fill = first same-coin tick with `ts ≥ signal+L`; gap reject if delay > L+1000 ms.


In [ ]:
path = EVENTS if EVENTS.exists() else SMOKE
assert path.exists(), "run: python3 -m research.gear22_signal_fill_lib"
ev = pd.read_parquet(path)
meta = json.loads((path.parent / "run_meta.json").read_text()) if (path.parent / "run_meta.json").exists() else {}
print("source", path)
print("rows", len(ev), "coins", ev["base_coin"].nunique(), "blocks", ev["block_id"].nunique())
print("L counts:\n", ev.groupby("L_ms").size())
print("meta keys", list(meta.keys())[:12])
d100 = ev.loc[ev["L_ms"] == PRIMARY_L]
print("effective_latency_ms L=100", d100["effective_latency_ms"].describe())


## Arms A/B/C — correlations (temporal weights + block bootstrap)


In [ ]:
summary_path = path.parent / f"summary_L{PRIMARY_L}.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
else:
    summary = summarize(ev, L=PRIMARY_L)
    summary_path.write_text(json.dumps(summary, indent=2, default=str))

verdict = verdict_from_summary(summary)
(path.parent / "verdict.txt").write_text(verdict + "\n")
print("VERDICT:", verdict)
print("arm_A", summary["arm_A"])
print("arm_B spearman x,delta", summary["arm_B"]["spearman_x_delta"])
print("arm_B absx,absdelta", summary["arm_B"]["spearman_absx_absdelta"])
print("arm_B equal coin/block", summary["arm_B"]["equal_coin_spearman_x_delta"], summary["arm_B"]["equal_block_spearman_x_delta"])
print("arm_C spearman z,delta", summary["arm_C"]["spearman_z_delta"])
print("positive_x", {k: summary["positive_x"][k] for k in ("n", "p_delta_neg", "median_delta", "median_correction", "median_R")})


## Conditional bins + sensitivity 50/100/200 ms


In [ ]:
bins = pd.DataFrame(summary.get("bins_x") or [])
display(bins[["bin", "n", "n_blocks", "median_x", "median_delta", "q10_delta", "q90_delta", "p_delta_neg_given_x_pos", "median_correction"]])

fig, ax = plt.subplots(figsize=(7, 4))
if len(bins):
    ax.plot(bins["median_x"], bins["median_delta"], marker="o", label="median Δ")
    ax.fill_between(bins["median_x"], bins["q10_delta"], bins["q90_delta"], alpha=0.2, label="Q10–Q90")
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("median x in bin")
ax.set_ylabel("Δ_L")
ax.set_title(f"Conditional Δ vs x (L={PRIMARY_L} ms)")
ax.legend()
plt.show()

for L in (50, 100, 200):
    sp = path.parent / f"summary_L{L}.json"
    if not sp.exists():
        sL = summarize(ev, L=L)
        sp.write_text(json.dumps(sL, indent=2, default=str))
    else:
        sL = json.loads(sp.read_text())
    print(f"L={L}", "spearman(x,Δ)=", sL["arm_B"]["spearman_x_delta"], "p_neg=", sL["positive_x"]["p_delta_neg"], "medΔ=", sL["positive_x"]["median_delta"])


## Arm D — trigger / freshness / side


In [ ]:
for k, sl in summary.get("arm_D_slices", {}).items():
    print(k, "n=", sl.get("n"), "spearman=", sl.get("spearman_x_delta"), "p_neg|x>0=", sl.get("p_delta_neg_given_x_pos"))
print("decomp_pos_x", summary.get("decomp_pos_x"))
print("top coins", summary.get("top_coins_by_abs_corr", [])[:8])

# Diagnostic only (not pre-registered verdict input): nonzero moves
pos = d100.loc[d100["x"] > 0]
nz = pos.loc[pos["delta"] != 0]
print("diag share nonzero|x>0", len(nz) / max(len(pos), 1))
if len(nz):
    print("diag P(Δ<0|x>0,Δ≠0)", float((nz["delta"] < 0).mean()))


## Stop

Канон / entry rules / handoff — **не** делать из этого ноутбука. Вердикт только в `research/gear22_signal_fill_response.md`.
